In [5]:
import pandas as pd
import tempfile
from tokenizers import BertWordPieceTokenizer
from transformers import BertTokenizerFast
from transformers import BertConfig
from transformers import BertForMaskedLM
from datasets import load_dataset
from transformers import DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments
from transformers import BertForSequenceClassification
from sklearn.preprocessing import LabelEncoder
from datasets import load_dataset
from sklearn.metrics import f1_score, accuracy_score
import numpy as np
import torch
import tqdm
from sklearn import model_selection
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import seaborn as sns
from torch import nn


In [2]:
comments = pd.read_csv('/kaggle/input/datasets/mcchainiy/comments-labels/comments.csv')

Обучаем токенайзер

In [ ]:
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    for comment in comments['comment']:
        f.write(comment + '\n')
    temp_path = f.name

In [ ]:
tokenizer = BertWordPieceTokenizer(lowercase=True)

tokenizer.train(
    files=[temp_path],
    vocab_size=30000,
    min_frequency=2,
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
)

In [ ]:
tokenizer.save_model("tokenizer")

In [ ]:
tokenizer.save("tokenizer/tokenizer.json")

Загружаем токенизатор

In [13]:
tokenizer = BertTokenizerFast.from_pretrained("./tokenizer")

Обучаем берт

In [ ]:
config = BertConfig(
    vocab_size=30000,
    hidden_size=256,
    num_hidden_layers=4,
    num_attention_heads=4,
    intermediate_size=512,
    max_position_embeddings=512
)

In [ ]:
model = BertForMaskedLM(config=config)

Dataset for mlm

In [ ]:
dataset = load_dataset(
    "text",
    data_files={"train": temp_path}
)

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

Маскируем датасет

In [ ]:

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

Обучаем модель

In [ ]:
training_args = TrainingArguments(
    output_dir="./custom_bert",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    learning_rate=5e-4,
    save_steps=1000,
    logging_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator
)

trainer.train()

In [ ]:
trainer.save_model("./bert")
tokenizer.save_pretrained("./bert")